First, I'll profile the efficiency of matrix multiplication. Then I'll compare this to the efficiency using TF32 and using automatic mixed precision (AMP).

In [ ]:
from time import time
import torch

In [ ]:
warmup = 5
n = 10
N = 128
D = 1024

In [ ]:
assert torch.cuda.is_available()

In [ ]:
device = torch.cuda.current_device()
device_index = torch.cuda.current_device()
print(f"Currently selected GPU: {device_index}")
print(f"Name of current GPU: {torch.cuda.get_device_name(device_index)}")

# Matrix Multiplication Runtime

### FP32 Precision

In [ ]:
X = torch.randn(N, D, D, device=device)

print(f"Data type: {X.dtype}")
print("----------------------")

print("Starting warmup")
for _ in range(warmup):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
print("Warmup complete")

start_time = time()
for _ in range(n):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("FP32 Multiplication")
print("----------------------")
print(f"Time per matmul: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*2*(D**3)/total_time/1e12:.2f} TFLOPs\n")

### TF32 Precision (19 bits)

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True 
torch.backends.cudnn.allow_tf32 = True

In [ ]:
X = torch.randn(N, D, D, device=device)

print(f"Data type: {X.dtype}")
print("----------------------")

print("Starting warmup")
for _ in range(warmup):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
print("Warmup complete")

start_time = time()
for _ in range(n):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("TF32 Multiplication")
print("----------------------")
print(f"Time per matmul: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*2*(D**3)/total_time/1e12:.2f} TFLOPs\n")

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

### FP16 Precision

In [ ]:
torch.set_default_dtype(torch.float16)

In [ ]:
X = torch.randn(N//2, D//2, D//2, device=device)

print(f"Data type: {X.dtype}")
print("----------------------")

print("Starting warmup")
for _ in range(warmup):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
print("Warmup complete")

start_time = time()
for _ in range(n):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("FP16 Multiplication")
print("----------------------")
print(f"Time per matmul: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N/2*2*(D/2**3)/total_time/1e12:.2f} TFLOPs\n")

In [ ]:
torch.set_default_dtype(torch.float32)

### BF16 Precision

In [ ]:
torch.set_default_dtype(torch.bfloat16)

In [ ]:
X = torch.randn(N, D, D, device=device)

print(f"Data type: {X.dtype}")
print("----------------------")

print("Starting warmup")
for _ in range(warmup):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
print("Warmup complete")

start_time = time()
for _ in range(n):
    C = torch.bmm(X, X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("BF16 Multiplication")
print("----------------------")
print(f"Time per matmul: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*2*(D**3)/total_time/1e12:.2f} TFLOPs\n")

In [ ]:
torch.set_default_dtype(torch.float32)

### Automatic Mixed Precision (AMP)

In [ ]:
X = torch.randn(N, D, D, device=device)

print(f"Data type: {X.dtype}")
print("----------------------")

with torch.autocast(device.type):

    print("Starting warmup")
    for _ in range(warmup):
        C = torch.bmm(X, X)
    torch.cuda.synchronize()
    print("Warmup complete")

    start_time = time()
    for _ in range(n):
        C = torch.bmm(X, X)
    torch.cuda.synchronize()
    end_time = time()
    total_time = end_time-start_time

print("Mixed Precision Multiplication")
print("----------------------")
print(f"Time per matmul: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*2*(D**3)/total_time/1e12:2f} TFLOPs\n")

In [ ]:
torch.set_default_dtype(torch.float32)

# Model Inference Runtime

In [ ]:
import torchvision

In [ ]:
model = torchvision.models.resnet50().to(device)
warmup = 2
n = 10
N = 8
C = 3
H, W = 224, 224

In [ ]:
model_flops = 4.09e9

### FP32

In [ ]:
X = torch.rand(N, C, H, W, device=device)
print(f"Data type: {X.dtype}")
print("Started warmup")
for _ in range(warmup):
    model(X)
torch.cuda.synchronize()
print("Finished warmup")

start_time = time()
for _ in range(n):
    model(X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("FP32 ResNet50")
print("----------------------")
print(f"Time per model evaluation: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*model_flops/total_time/1e12:2f} TFLOPs\n")

### FP16

In [ ]:
torch.set_default_dtype(torch.float16)

In [ ]:
X = torch.rand(N, C, H, W, device=device)
print(f"Data type: {X.dtype}")
print("Started warmup")
for _ in range(warmup):
    model(X)
torch.cuda.synchronize()
print("Finished warmup")

start_time = time()
for _ in range(n):
    model(X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("FP16 ResNet50")
print("----------------------")
print(f"Time per model evaluation: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*model_flops/total_time/1e12:2f} TFLOPs\n")

### BF16

In [ ]:
torch.set_default_dtype(torch.bfloat16)

In [ ]:
X = torch.rand(N, C, H, W, device=device)
print(f"Data type: {X.dtype}")
print("Started warmup")
for _ in range(warmup):
    model(X)
torch.cuda.synchronize()
print("Finished warmup")

start_time = time()
for _ in range(n):
    model(X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("BF16 ResNet50")
print("----------------------")
print(f"Time per model evaluation: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*model_flops/total_time/1e12:2f} TFLOPs\n")

In [ ]:
torch.set_default_dtype(torch.float32)

### TF32

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True 
torch.backends.cudnn.allow_tf32 = True

In [ ]:
X = torch.rand(N, C, H, W, device=device)
print(f"Data type: {X.dtype}")
print("Started warmup")
for _ in range(warmup):
    model(X)
torch.cuda.synchronize()
print("Finished warmup")

start_time = time()
for _ in range(n):
    model(X)
torch.cuda.synchronize()
end_time = time()
total_time = end_time-start_time

print("TF32 ResNet50")
print("----------------------")
print(f"Time per model evaluation: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*model_flops/total_time/1e12:2f} TFLOPs\n")

### AMP

In [ ]:
X = torch.rand(N, C, H, W, device=device)
print(f"Data type: {X.dtype}")
print("Started warmup")
with torch.autocast(device.type):
    for _ in range(warmup):
        model(X)
    torch.cuda.synchronize()
    print("Finished warmup")

    start_time = time()
    for _ in range(n):
        model(X)
    torch.cuda.synchronize()
    end_time = time()
    total_time = end_time-start_time

print("Mixed Precision ResNet50")
print("----------------------")
print(f"Time per model evaluation: {(total_time)/n:.3f} seconds")
print(f"Flops: {n*N*model_flops/total_time/1e12:2f} TFLOPs\n")